# Pi0 Comprehensive Study

This notebook implements three complementary experiments on the Pi0 model:

1. **Baseline Benchmarking**: Record unmodified performance on LIBERO, VLABench, MetaWorld
2. **Ablation Study (Pass0)**: Zero state encoder (`state_proj`) and measure success rate drops
3. **Gradient Analysis**: Measure state encoder gradient contributions using flow matching loss

## Model Details
- **State Encoder**: `state_proj` (Single Linear layer)
- **Loss Function**: Flow matching `F.mse_loss(u_t, v_t)` from Physical-Intelligence/openpi
- **Benchmarks**: LIBERO + VLABench + MetaWorld

## Expected Results
- Baseline provides reference success rates
- Ablation shows performance impact (success rate drop)
- Gradient analysis shows training signal strength (gradient magnitude)
- Cross-validation: Gradient ∝ Performance impact?

---
## Part 1 · Comprehensive Study

Uses [Physical-Intelligence/openpi](https://github.com/Physical-Intelligence/openpi) (JAX server) for LIBERO,
[lerobot](https://github.com/huggingface/lerobot) for MetaWorld,
and [Shiduo-zh/openpi](https://github.com/Shiduo-zh/openpi) (JAX server) for VLABench.

| Benchmark | Checkpoint | Notes |
|-----------|-----------|-------|
| LIBERO | `pi05_libero` (openpi) | pi0.5 JAX checkpoint from GCS, served via openpi WebSocket |
| MetaWorld | `lerobot/pi0_old` | Original generalist Pi0 (no openpi checkpoint) — negative control |
| VLABench | `VLABench/pi0-primitive-10task` | Fine-tuned pi0 JAX orbax checkpoint, served via Shiduo-zh/openpi WebSocket |

**Ablation method** (consistent across all benchmarks):  
Zero the **output** of `state_proj` so no proprioceptive information reaches the expert model. The zeroed tensor has the same shape as the unablated state encoder output.

- **LIBERO / VLABench** (JAX server): `serve_policy_ablated.py` monkey-patches `Policy.__init__` to subclass `state_proj`'s Linear module and override `__call__` to return `zeros_like(output)` before the JIT closure is compiled — state encoder output is exactly zero with correct shape for all inputs.  
  > **Note**: `pi05_libero` uses the pi0.5 architecture, which replaces `state_proj` with `time_mlp_in`/`time_mlp_out`. The ablation patch logs a warning if `state_proj` is not found.  
- **MetaWorld** (PyTorch): `register_forward_hook` returns `zeros_like(output)` — fires after the Linear forward pass, guaranteeing exact zero embedding with correct shape regardless of input.


In [1]:
# ── Paths & workspace ──────────────────────────────────────────────────────
from google.colab import drive, auth as _colab_auth
from pathlib import Path
import os, json, subprocess, sys, torch

drive.mount('/content/drive')
_colab_auth.authenticate_user()       # needed for GCS (VLABench ckpt if used)

WORKSPACE    = '/content/drive/MyDrive/pi0_study'
BASELINE_DIR = Path(f'{WORKSPACE}/Results/baseline')
ABLATION_DIR = Path(f'{WORKSPACE}/Results/ablation')
GRADIENT_DIR = Path(f'{WORKSPACE}/Results/gradient')
LOGS_DIR     = Path('/content/logs')

for d in [BASELINE_DIR, ABLATION_DIR, GRADIENT_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['MUJOCO_GL'] = 'egl'
print('✅ Paths ready')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths ready


In [ ]:
# ── Install all dependencies (run once, then restart runtime) ──────────────
# Architecture:
#   LIBERO   → Physical-Intelligence/openpi JAX server + Python 3.8 LIBERO venv client
#   MetaWorld → lerobot/pi0_old (no openpi checkpoint exists) — negative control
#   VLABench → Shiduo-zh/openpi JAX server + VLABench Python client
import subprocess, sys, os
from pathlib import Path

# ── 1. uv (manages openpi JAX venvs) ─────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
print('✅ uv installed')

# ── 2. Physical-Intelligence/openpi (LIBERO JAX server) ──────────────────
OPENPI_DIR = '/content/openpi'
if not Path(OPENPI_DIR).exists():
    print('⏳ Cloning Physical-Intelligence/openpi (with submodules)...')
    subprocess.run(
        ['git', 'clone', '--quiet', '--recurse-submodules',
         'https://github.com/Physical-Intelligence/openpi.git', OPENPI_DIR],
        env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}, check=True)
    print('✅ openpi cloned')
else:
    print('✅ openpi already cloned')

# Sync openpi JAX server uv environment (downloads JAX, orbax, flax, etc.)
print('⏳ Syncing openpi uv environment (JAX download, ~1-2 min)...')
r = subprocess.run(
    ['uv', 'sync', '--project', OPENPI_DIR],
    capture_output=True, text=True,
    env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
if r.returncode != 0:
    print('⚠️  uv sync stderr:', r.stderr[-2000:])
    raise RuntimeError('openpi uv sync failed')
print('✅ openpi uv environment synced')

# ── 3. LIBERO client venv (Python 3.8) ───────────────────────────────────
# examples/libero/main.py runs LIBERO environments locally and calls the JAX
# server via WebSocket for policy inference.  robosuite/LIBERO require Py 3.8.
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'

if not Path(LIBERO_PY).exists():
    print('⏳ Creating LIBERO client venv (Python 3.8)...')
    subprocess.run(['uv', 'venv', '--python', '3.8', LIBERO_VENV], check=True)
    # Install compiled requirements (torch 1.11+cu113 from PyTorch wheel index)
    subprocess.run(
        ['uv', 'pip', 'sync',
         '--python', LIBERO_PY,
         f'{OPENPI_DIR}/examples/libero/requirements.txt',
         f'{OPENPI_DIR}/third_party/libero/requirements.txt',
         '--extra-index-url', 'https://download.pytorch.org/whl/cu113',
         '--index-strategy=unsafe-best-match'],
        check=True)
    # openpi-client WebSocket package + local LIBERO editable install
    subprocess.run(
        ['uv', 'pip', 'install', '--python', LIBERO_PY,
         '-e', f'{OPENPI_DIR}/packages/openpi-client',
         '-e', f'{OPENPI_DIR}/third_party/libero'],
        check=True)
    print('✅ LIBERO client venv ready')
else:
    print('✅ LIBERO client venv already exists')

# ── 4. Shiduo-zh/openpi (VLABench JAX server) ───────────────────────────
OPENPI_VLA = '/content/openpi_vlabench'
if not Path(OPENPI_VLA).exists():
    print('⏳ Cloning Shiduo-zh/openpi (VLABench fork)...')
    subprocess.run(
        ['git', 'clone', '--quiet',
         'https://github.com/Shiduo-zh/openpi.git', OPENPI_VLA],
        env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}, check=True)
    print('✅ Shiduo-zh/openpi cloned')
else:
    print('✅ Shiduo-zh/openpi already cloned')

print('⏳ Syncing VLABench openpi uv environment...')
r = subprocess.run(
    ['uv', 'sync', '--project', OPENPI_VLA],
    capture_output=True, text=True,
    env={**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'})
if r.returncode != 0:
    print('⚠️  uv sync stderr:', r.stderr[-2000:])
    raise RuntimeError('VLABench openpi uv sync failed')
print('✅ VLABench openpi uv environment synced')

# ── 5. VLABench Python client (Evaluator + gym envs + OpenPiPolicy) ──────
if not Path('/content/VLABench').exists():
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/OpenDriveLab/VLABench.git',
                    '/content/VLABench'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-e', '/content/VLABench'], check=True)
print('✅ VLABench installed')

# ── 6. lerobot + transformers fork (MetaWorld + Part 2 gradient analysis) ──
# MetaWorld uses lerobot/pi0_old — no openpi checkpoint exists for MetaWorld.
# lerobot[metaworld,pi] includes MetaWorld envs + Pi0Policy weights.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lerobot[metaworld,pi] @ git+https://github.com/huggingface/lerobot.git'],
               check=True)
# Pi0Policy requires APIs (embed_image, embed_language_tokens) from the
# lerobot-maintained transformers branch (4.53.x).  Vanilla 4.51/4.52 fails.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers @ git+https://github.com/huggingface/transformers.git@fix/lerobot_openpi'],
               check=True)
# websockets + msgpack for VLABench WebSocket client
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'websockets', 'msgpack'], check=True)
print('✅ lerobot[metaworld] + transformers fork + websockets installed')

# ── HuggingFace login ──────────────────────────────────────────────────────
# Required for VLABench/pi0-primitive-10task checkpoint and google/gemma-2b (Part 2).
from huggingface_hub import login as _hf_login
_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab secrets')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')
    if _hf_token:
        print('✅ HF_TOKEN loaded from environment')
if not _hf_token:
    import getpass
    _hf_token = getpass.getpass('🔑 HuggingFace token (hidden): ')
if _hf_token:
    _hf_login(token=_hf_token, add_to_git_credential=False)
    print('✅ HuggingFace login OK')
else:
    raise RuntimeError('No HF token — needed for VLABench checkpoint + Part 2 tokenizer.')

print('\n✅ Install complete — restart runtime, then run all cells.')

In [ ]:
# ── Shared helpers — policy loading, ablation hook, tokeniser ──────────────
# Used by MetaWorld (lerobot/pi0_old) and Part 2 gradient analysis.
# LIBERO and VLABench use openpi JAX servers — no PyTorch policy loading needed.
import torch
from lerobot.policies.pi0.modeling_pi0 import PI0Policy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# lerobot/pi0_old — original generalist Pi0 checkpoint (used for MetaWorld).
# No openpi checkpoint exists for MetaWorld; this serves as a negative control.
POLICY_ID = 'lerobot/pi0_old'

def load_policy(policy_id=POLICY_ID):
    return PI0Policy.from_pretrained(policy_id).to(DEVICE).eval()

def hook_state_proj(policy):
    """Register a forward hook that zeros state_proj output. Returns handles.

    Used in MetaWorld ablation (Part 1) and gradient analysis (Part 2).
    LIBERO and VLABench ablations are done server-side in serve_policy_ablated.py.
    """
    handles = []
    for name, module in policy.named_modules():
        if 'state_proj' in name and isinstance(module, torch.nn.Linear):
            h = module.register_forward_hook(lambda m, i, o: torch.zeros_like(o))
            handles.append(h)
            print(f'  ↳ Zeroing: {name}')
    if not handles:
        raise RuntimeError('state_proj not found — wrong checkpoint?')
    return handles

def get_pi0_tokenizer():
    """Load the Gemma tokenizer for Pi0.

    Pi0 is built on PaliGemma (Gemma-2B backbone), so the correct tokenizer
    is google/gemma-2b. Neither lerobot/pi0 nor lerobot/pi0_old ship
    a tokenizer — it must be loaded from the source model.

    Requires: HF login + accept licence at https://huggingface.co/google/gemma-2b
    """
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('google/gemma-2b')
    print(f'  Tokenizer: AutoTokenizer from google/gemma-2b')
    return tok

print(f'✅ Helpers ready  (device: {DEVICE})')
print(f'   MetaWorld policy: {POLICY_ID}')

---
## 2. LIBERO Benchmark — `pi05_libero` (openpi JAX server)

Runs the **Physical-Intelligence/openpi** JAX server with the `pi05_libero` checkpoint
(pi0.5 architecture, checkpoint at `gs://openpi-assets/checkpoints/pi05_libero`).

The **LIBERO client** (`examples/libero/main.py`) runs in a dedicated Python 3.8 virtual
environment with robosuite 1.4.1 and connects to the JAX server via WebSocket.

| Suite | Tasks | Episodes per task |
|-------|-------|-------------------|
| libero_spatial | 10 | 5 |
| libero_object | 10 | 5 |
| libero_goal | 10 | 5 |
| libero_90 | 90 | 5 |

Results are logged as `"Total success rate: X.XXXXX"` by the client and collected per-suite.

In [ ]:
# ── LIBERO Baseline ────────────────────────────────────────────────────────
# Architecture:
#   Server: uv run scripts/serve_policy.py --env libero   (pi05_libero from GCS)
#   Client: examples/libero/.venv/bin/python examples/libero/main.py  (Py 3.8 venv)
#
# The openpi server downloads pi05_libero from GCS on first run (~5-10 min).
# The client logs "Total success rate: X.XXXXX" which we parse per-suite.
# PYTHONPATH must include third_party/libero for the LIBERO gym environments.
import os, json, re, subprocess, socket, time
from pathlib import Path

OPENPI_DIR  = '/content/openpi'
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'
LIBERO_MAIN = f'{OPENPI_DIR}/examples/libero/main.py'
SERVER_PORT = 8000
N_TRIALS    = 5   # trials per task  (10 tasks × 5 = 50 total per suite)

SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']

# Start openpi LIBERO server (pi05_libero checkpoint pulled from GCS)
print('🚀 Starting openpi LIBERO server (pi05_libero from GCS, first run ~10 min)...')
server_proc = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_DIR,
     'python', f'{OPENPI_DIR}/scripts/serve_policy.py',
     '--env', 'libero',
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_DIR,
    env={**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'  PID {server_proc.pid} — waiting for port {SERVER_PORT}...')

# Poll up to 10 min (GCS checkpoint download on first run)
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(5)
else:
    server_proc.kill()
    out, _ = server_proc.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'openpi LIBERO server did not open port {SERVER_PORT} within 10 minutes')
print(f'✅ openpi LIBERO server ready on port {SERVER_PORT}')

# Evaluate each suite using the LIBERO venv client
libero_baseline = {}
libero_env = {**os.environ,
              'PYTHONPATH': f'{OPENPI_DIR}/third_party/libero:{os.environ.get("PYTHONPATH", "")}',
              'MUJOCO_GL': 'egl'}

for suite in SUITES:
    print(f'\n📍 Baseline: {suite}')
    result = subprocess.run(
        [LIBERO_PY, LIBERO_MAIN,
         '--task-suite-name', suite,
         '--num-trials-per-task', str(N_TRIALS),
         '--host', 'localhost',
         '--port', str(SERVER_PORT)],
        capture_output=True, text=True,
        env=libero_env,
    )
    combined = result.stdout + result.stderr
    match = re.search(r'Total success rate:\s*([\d.]+)', combined)
    if not match:
        print('Client output:\n', combined[-3000:])
        raise RuntimeError(f'Could not parse "Total success rate" for {suite}')
    sr = float(match.group(1))
    libero_baseline[suite] = {'success_rate': sr}
    print(f'  ✅ {suite} SR: {sr:.1%}')

# Stop server
server_proc.terminate()
server_proc.wait()
print('\n🛑 openpi LIBERO server stopped')

with open(BASELINE_DIR / 'libero_baseline.json', 'w') as f:
    json.dump(libero_baseline, f, indent=2)
overall = sum(r['success_rate'] for r in libero_baseline.values()) / len(libero_baseline)
print(f'✅ LIBERO baseline done  |  Overall SR: {overall:.1%}')

In [ ]:
# ── LIBERO Ablation ────────────────────────────────────────────────────────
# Writes serve_policy_ablated.py into the openpi scripts directory by prepending
# a monkey-patch that patches state_proj.__call__ to return zeros_like(output)
# before Policy.__init__ compiles the JIT closure.
# The same LIBERO venv client is used as in the baseline.
#
# Note: pi05_libero uses pi0.5 architecture (state_proj replaced by
# time_mlp_in/time_mlp_out).  The patch prints a warning if state_proj is not
# found — in that case the "ablated" result equals baseline (expected for pi0.5).
import os, json, re, subprocess, socket, time
from pathlib import Path

OPENPI_DIR  = '/content/openpi'
LIBERO_VENV = f'{OPENPI_DIR}/examples/libero/.venv'
LIBERO_PY   = f'{LIBERO_VENV}/bin/python'
LIBERO_MAIN = f'{OPENPI_DIR}/examples/libero/main.py'
SERVER_PORT = 8000
N_TRIALS    = 5

SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90']

# Write ablated server script (ablation prefix + original serve_policy.py)
_serve_src = open(f'{OPENPI_DIR}/scripts/serve_policy.py').read()

_ablation_prefix = '''\
# ── ABLATION PATCH (prepended to serve_policy.py) ────────────────────────
# Patches state_proj.__call__ to return zeros_like(output) before
# Policy.__init__ compiles the JIT closure.  Policy does NOT keep a model
# reference after __init__, so we must intercept the model here.
# Zeroing the OUTPUT (not the weights) preserves the correct output shape
# while ensuring no state information reaches the expert model.
#
# pi0.5 note: pi05_libero uses time_mlp_in/time_mlp_out instead of state_proj.
# If state_proj is not found, a warning is printed and ablation has no effect.
import openpi.policies.policy as _pol_mod
import jax.numpy as _jnp

_orig_policy_init = _pol_mod.Policy.__init__

def _find_and_patch_state_proj(model, depth: int = 6) -> bool:
    if depth <= 0:
        return False
    if hasattr(model, "state_proj"):
        sp = model.state_proj
        _orig_call = sp.__class__.__call__
        # Subclass the Linear module so this instance returns zeros_like(output)
        sp.__class__ = type(
            "ZeroOutputLinear",
            (sp.__class__,),
            {"__call__": lambda self, x: _jnp.zeros_like(_orig_call(self, x))},
        )
        print("[ABLATION] state_proj output zeroed (zeros_like) \u2705")
        return True
    for attr in ("model", "_model", "pi0", "transformer"):
        child = getattr(model, attr, None)
        if child is not None and _find_and_patch_state_proj(child, depth - 1):
            return True
    return False

def _ablated_policy_init(self, model, *args, **kwargs):
    found = _find_and_patch_state_proj(model)
    if not found:
        print("[ABLATION] WARNING: state_proj not found (pi0.5 uses time_mlp instead) — no ablation applied")
    _orig_policy_init(self, model, *args, **kwargs)

_pol_mod.Policy.__init__ = _ablated_policy_init
print("[ABLATION] Policy.__init__ monkey-patched — state_proj output zeroed on load")
# ── END ABLATION PATCH ────────────────────────────────────────────────────

'''

_ablated_path = f'{OPENPI_DIR}/scripts/serve_policy_ablated.py'
with open(_ablated_path, 'w') as f:
    f.write(_ablation_prefix + _serve_src)
print(f'✅ serve_policy_ablated.py written to {_ablated_path}')

# Start ablated openpi LIBERO server
print('🚀 Starting ablated openpi LIBERO server...')
server_proc = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_DIR,
     'python', f'{OPENPI_DIR}/scripts/serve_policy_ablated.py',
     '--env', 'libero',
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_DIR,
    env={**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'  PID {server_proc.pid} — waiting for port {SERVER_PORT}...')

deadline = time.time() + 600
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(5)
else:
    server_proc.kill()
    out, _ = server_proc.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'Ablated openpi LIBERO server did not open port {SERVER_PORT} within 10 minutes')
print(f'✅ Ablated openpi LIBERO server ready on port {SERVER_PORT}')

# Run evaluation with LIBERO venv client (same as baseline)
libero_ablation = {}
libero_env = {**os.environ,
              'PYTHONPATH': f'{OPENPI_DIR}/third_party/libero:{os.environ.get("PYTHONPATH", "")}',
              'MUJOCO_GL': 'egl'}

for suite in SUITES:
    print(f'\n📍 Ablation: {suite}')
    result = subprocess.run(
        [LIBERO_PY, LIBERO_MAIN,
         '--task-suite-name', suite,
         '--num-trials-per-task', str(N_TRIALS),
         '--host', 'localhost',
         '--port', str(SERVER_PORT)],
        capture_output=True, text=True,
        env=libero_env,
    )
    combined = result.stdout + result.stderr
    match = re.search(r'Total success rate:\s*([\d.]+)', combined)
    if not match:
        print('Client output:\n', combined[-3000:])
        raise RuntimeError(f'Could not parse "Total success rate" for {suite}')
    sr = float(match.group(1))
    libero_ablation[suite] = {'success_rate': sr}
    print(f'  ✅ {suite} SR (ablated): {sr:.1%}')

# Stop server
server_proc.terminate()
server_proc.wait()
print('\n🛑 Ablated openpi LIBERO server stopped')

with open(ABLATION_DIR / 'libero_ablation.json', 'w') as f:
    json.dump(libero_ablation, f, indent=2)
print('✅ LIBERO ablation done')
print()
print('⚠️  CAVEAT: pi05_libero uses the pi0.5 architecture which replaces state_proj')
print('   with time_mlp_in/time_mlp_out. If state_proj was not found by the ablation')
print('   patch, the ablated run is IDENTICAL to baseline (no-op ablation).')
print('   A 0% success rate drop for LIBERO is expected in this case and is NOT')
print('   evidence that proprioceptive state is unused — it means the ablation')
print('   could not target the correct module in pi0.5 architecture.')
print('   To properly ablate pi0.5, target time_mlp_in/time_mlp_out instead.')


---
## 3. MetaWorld — `lerobot/pi0_droid` (generalist)

`lerobot/pi0_old` is the original generalist π₀ checkpoint (previously named `lerobot/pi0`).
No MetaWorld-finetuned checkpoint exists; **~0% SR is expected on both baseline and ablated runs**.
This is included for research completeness — confirming that state_proj has no measurable effect
when the model is already near-chance (out-of-distribution).

Ablation: zero `observation.state` output (matches Evo-1 and LIBERO methodology).

In [ ]:
# -- MetaWorld Baseline -------------------------------------------------------
# Runs MetaWorld evaluation in a subprocess -- child process owns all VRAM.
# lerobot-eval does not support MetaWorld directly, so mw_eval.py is a
# self-contained script with baseline + ablation (--ablate flag) in one file.
#
# lerobot/pi0_old is the original generalist pi0; ~0% SR expected (negative ctrl).
import os, subprocess, sys, json
from pathlib import Path

MW_EVAL = Path('/content/mw_eval.py')
MW_EVAL.write_text('"""MetaWorld evaluation script (baseline + ablation via --ablate flag).\n\nRun as:\n  python mw_eval.py --policy lerobot/pi0_old --out /path/results.json\n  python mw_eval.py --policy lerobot/pi0_old --out /path/results.json --ablate\n"""\nimport json, argparse\nimport numpy as np, torch\nimport metaworld\nfrom pathlib import Path\nfrom lerobot.policies.pi0.modeling_pi0 import PI0Policy\nfrom transformers import AutoTokenizer\n\n\ndef run_eval(policy_id, output_path, task_names, n_episodes, max_steps, ablate=False):\n    device    = "cuda" if torch.cuda.is_available() else "cpu"\n    policy    = PI0Policy.from_pretrained(policy_id).to(device).eval()\n    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")\n\n    if ablate:\n        count = 0\n        for name, module in policy.named_modules():\n            if "state_proj" in name and isinstance(module, torch.nn.Linear):\n                module.register_forward_hook(lambda m, i, o: torch.zeros_like(o))\n                count += 1\n                print(f"[ABLATION] Zeroed output of {name}", flush=True)\n        if count == 0:\n            raise RuntimeError("[ABLATION] state_proj not found!")\n\n    dummy_imgs = {}\n    if hasattr(policy.config, "input_features"):\n        for k in policy.config.input_features:\n            if "images" in k:\n                dummy_imgs[k] = torch.zeros(1, 3, 224, 224, device=device)\n\n    results = {}\n    for task_name in task_names:\n        print(f"\\nTask: {task_name}", flush=True)\n        ml1 = metaworld.ML1(task_name)\n        env = ml1.train_classes[task_name]()\n        env.set_task(next(iter(ml1.train_tasks)))\n        instruction = task_name.replace("-v2", "").replace("-", " ")\n        enc = tokenizer(instruction, return_tensors="pt", padding="max_length",\n                        max_length=policy.config.tokenizer_max_length, truncation=True)\n        lang_ids  = enc["input_ids"].to(device)\n        lang_mask = enc["attention_mask"].to(device)\n\n        successes = []\n        for _ in range(n_episodes):\n            try:    obs, _ = env.reset()\n            except: obs    = env.reset()\n            policy.reset()\n            for _ in range(max_steps):\n                obs_t = torch.tensor(\n                    obs if isinstance(obs, np.ndarray) else obs["state"],\n                    dtype=torch.float32).unsqueeze(0).to(device)\n                batch = {"observation.state": obs_t,\n                         "observation.language.tokens": lang_ids,\n                         "observation.language.attention_mask": lang_mask,\n                         **dummy_imgs}\n                with torch.no_grad():\n                    action = policy.select_action(batch).cpu().numpy()\n                if action.ndim > 1: action = action[0]\n                try:    obs, _, terminated, truncated, info = env.step(action)\n                except: obs, _, done, info = env.step(action); terminated = done; truncated = False\n                if info.get("success", False): successes.append(True); break\n                if terminated or truncated:    successes.append(False); break\n            else: successes.append(False)\n\n        env.close()\n        sr = sum(successes) / len(successes)\n        results[task_name] = {"success_rate": sr}\n        print(f"  SR: {sr:.1%}", flush=True)\n\n    Path(output_path).parent.mkdir(parents=True, exist_ok=True)\n    with open(output_path, "w") as f:\n        json.dump(results, f, indent=2)\n    print(f"\\nSaved: {output_path}", flush=True)\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--policy",    required=True)\n    p.add_argument("--out",       required=True)\n    p.add_argument("--n-eps",     type=int, default=10)\n    p.add_argument("--max-steps", type=int, default=500)\n    p.add_argument("--ablate",    action="store_true")\n    a = p.parse_args()\n    tasks = ["reach-v2", "push-v2", "pick-place-v2", "door-open-v2", "drawer-close-v2"]\n    run_eval(a.policy, a.out, tasks, a.n_eps, a.max_steps, ablate=a.ablate)\n')
print(f'MetaWorld eval script written: {MW_EVAL}')

MW_EPISODES = 10
output_path = BASELINE_DIR / 'metaworld_baseline.json'

print('\nRunning MetaWorld baseline (subprocess)...')
subprocess.run([
    sys.executable, str(MW_EVAL),
    '--policy', 'lerobot/pi0_old',
    '--out',    str(output_path),
    '--n-eps',  str(MW_EPISODES),
], env={**os.environ, 'MUJOCO_GL': 'egl'}, check=True)

mw_baseline = json.loads(output_path.read_text())
for task, data in mw_baseline.items():
    print(f'  {task}: {data["success_rate"]:.1%}')
print('\n MetaWorld baseline done (~0% SR expected -- negative control)')


In [ ]:
# -- MetaWorld Ablation -------------------------------------------------------
# ABLATION METHOD: output-zeroing of state_proj via --ablate flag in mw_eval.py.
# ~0% SR on both baseline and ablated is expected (negative control).
import os, subprocess, sys, json
from pathlib import Path

MW_EVAL     = Path('/content/mw_eval.py')   # written in cell above
MW_EPISODES = 10                             # fallback if cell above wasn't run
output_path = ABLATION_DIR / 'metaworld_ablation.json'

print('Running MetaWorld ablation (subprocess)...')
subprocess.run([
    sys.executable, str(MW_EVAL),
    '--policy', 'lerobot/pi0_old',
    '--out',    str(output_path),
    '--n-eps',  str(MW_EPISODES),
    '--ablate',
], env={**os.environ, 'MUJOCO_GL': 'egl'}, check=True)

mw_ablation = json.loads(output_path.read_text())
for task, data in mw_ablation.items():
    print(f'  {task}: {data["success_rate"]:.1%}')
print('\n MetaWorld ablation done')


---
## 4. VLABench — `VLABench/pi0-primitive-10task` (fine-tuned, JAX server)

The fine-tuned checkpoint (`VLABench/pi0-primitive-10task`, ~43 GB orbax, loads ~8 GB at BF16)
is served by `Shiduo-zh/openpi` — a VLABench-compatible fork of Physical-Intelligence/openpi.
The server runs in its own isolated uv environment (JAX, orbax), freeing the Colab kernel from
any JAX dependency.

**Evaluation**: VLABench's own `Evaluator` + `OpenPiPolicy` WebSocket client handle env stepping,
observation construction, and success tracking.

**Ablation**: `AblatedOpenPiPolicy` zeros `ee_state` (end-effector state) before each WebSocket
request — consistent with input-zeroing used in LIBERO and MetaWorld.

In [ ]:
# ── VLABench: Download checkpoint + write ablated server ──────────────────
# Shiduo-zh/openpi was cloned and uv-synced in the install cell (Cell 3).
# This cell downloads the VLABench/pi0-primitive-10task checkpoint from
# HuggingFace and writes serve_policy_ablated.py into the cloned repo.
import os, subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

OPENPI_VLA    = '/content/openpi_vlabench'
VLABENCH_CKPT = '/content/ckpt_pi0_vlabench'

# Verify the VLABench openpi repo is present (cloned in install cell)
if not Path(f'{OPENPI_VLA}/scripts/serve_policy.py').exists():
    raise RuntimeError(
        f'Shiduo-zh/openpi not found at {OPENPI_VLA}. '
        'Run the install cell (Cell 3) first.')
print(f'✅ Shiduo-zh/openpi found at {OPENPI_VLA}')

# Download fine-tuned checkpoint from HuggingFace (~43 GB on disk)
if not (Path(VLABENCH_CKPT).exists() and os.listdir(VLABENCH_CKPT)):
    print('⬇️  Downloading VLABench/pi0-primitive-10task checkpoint (~43 GB)...')
    snapshot_download(
        repo_id='VLABench/pi0-primitive-10task',
        local_dir=VLABENCH_CKPT,
        ignore_patterns=['*.md'],
    )
    print('✅ Checkpoint downloaded')
else:
    print('✅ Checkpoint already downloaded')
print(f'Checkpoint at: {VLABENCH_CKPT}')

# ── Write serve_policy_ablated.py into the VLABench openpi repo ───────────
# Patches state_proj.__call__ to return zeros_like(output) before
# Policy.__init__ compiles the JIT closure.
#
# Methodology (consistent across all benchmarks):
#   PyTorch (MetaWorld): register_forward_hook returns zeros_like(output)
#   JAX (LIBERO/VLABench): state_proj.__call__ patched to return zeros_like(output)
#   Both guarantee the state embedding reaching the expert model is exactly 0
#   with the same shape as the original state encoder output.

_serve_policy_src = open(f'{OPENPI_VLA}/scripts/serve_policy.py').read()

_ablation_prefix = '''\
# ── ABLATION PATCH (prepended to serve_policy.py) ────────────────────────
# Patches state_proj.__call__ to return zeros_like(output) before
# Policy.__init__ compiles the JIT closure.  Policy does NOT keep a model
# reference after __init__, so we must intercept the model here.
# Zeroing the OUTPUT (not the weights) preserves the correct output shape
# while ensuring no state information reaches the expert model.
import openpi.policies.policy as _pol_mod
import jax.numpy as _jnp

_orig_policy_init = _pol_mod.Policy.__init__

def _find_and_patch_state_proj(model, depth: int = 6) -> bool:
    if depth <= 0:
        return False
    if hasattr(model, "state_proj"):
        sp = model.state_proj
        _orig_call = sp.__class__.__call__
        # Subclass the Linear module so this instance returns zeros_like(output)
        sp.__class__ = type(
            "ZeroOutputLinear",
            (sp.__class__,),
            {"__call__": lambda self, x: _jnp.zeros_like(_orig_call(self, x))},
        )
        print("[ABLATION] state_proj output zeroed (zeros_like) \u2705")
        return True
    for attr in ("model", "_model", "pi0", "transformer"):
        child = getattr(model, attr, None)
        if child is not None and _find_and_patch_state_proj(child, depth - 1):
            return True
    return False

def _ablated_policy_init(self, model, *args, **kwargs):
    found = _find_and_patch_state_proj(model)
    if not found:
        print("[ABLATION] WARNING: state_proj not found — ablation NOT applied!")
    _orig_policy_init(self, model, *args, **kwargs)

_pol_mod.Policy.__init__ = _ablated_policy_init
print("[ABLATION] Policy.__init__ monkey-patched — state_proj output zeroed on load")
# ── END ABLATION PATCH ────────────────────────────────────────────────────

'''

_ablated_path = f'{OPENPI_VLA}/scripts/serve_policy_ablated.py'
with open(_ablated_path, 'w') as _f:
    _f.write(_ablation_prefix + _serve_policy_src)
print(f'✅ serve_policy_ablated.py written to {_ablated_path}')


In [ ]:
# ── VLABench: Start openpi JAX policy server in background ────────────────
# serve_policy.py uses tyro CLI. The policy field is typed as Checkpoint | Default.
# Providing --policy.config and --policy.dir causes tyro to select the Checkpoint
# branch (Default has no fields). --env vlabench sets the environment mode.
#
# XLA_PYTHON_CLIENT_MEM_FRACTION=0.85 → ~34 GB on A100-40GB, ~68 GB on A100-80GB.
# The VLABench checkpoint loads at BF16 (~8 GB GPU), leaving plenty of headroom.
import os, subprocess, socket, time

OPENPI_VLA    = '/content/openpi_vlabench'
VLABENCH_CKPT = '/content/ckpt_pi0_vlabench'
SERVER_PORT   = 8000

env = {
    **os.environ,
    'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85',
}

server_proc = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_VLA,
     'python', f'{OPENPI_VLA}/scripts/serve_policy.py',
     '--env', 'vlabench',
     '--policy.config', 'pi0_ft_vlabench_primitive',
     '--policy.dir', VLABENCH_CKPT,
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_VLA,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'🚀 openpi VLABench server started (PID {server_proc.pid}), waiting for port {SERVER_PORT}...')

# Poll until the server's WebSocket port is open (timeout 5 min)
deadline = time.time() + 300
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(3)
else:
    server_proc.kill()
    # Print last lines of server stdout for debugging
    out, _ = server_proc.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'openpi VLABench server did not open port {SERVER_PORT} within 5 minutes')

print(f'✅ openpi VLABench server ready on port {SERVER_PORT}')

In [ ]:
# ── VLABench Baseline ─────────────────────────────────────────────────────
# Uses VLABench's own OpenPiPolicy (WebSocket client → background JAX server).
# OpenPiPolicy.predict() expects observation dict with keys:
#   "rgb"         – (right, left, front, wrist) images as numpy arrays
#   "ee_state"    – end-effector state (pos + quat/euler + gripper) float32
#   "instruction" – language instruction string
# The Evaluator class handles env stepping, obs construction, and success tracking.
import json, sys

sys.path.insert(0, '/content/VLABench')
# Direct import — OpenPiPolicy is NOT re-exported in __init__.py
from VLABench.evaluation.model.policy.openpi import OpenPiPolicy
from VLABench.evaluation.evaluator import Evaluator

VLA_TASKS    = ['select_toy', 'stack_block', 'put_in_drawer', 'close_drawer',
                'insert_peg', 'press_button', 'turn_tap', 'sweep_to_dustpan',
                'pick_and_place', 'reach_target']
VLA_EPISODES = 5
MAX_STEPS    = 300

policy = OpenPiPolicy(host='localhost', port=8000)

vla_baseline = {}
for task_name in VLA_TASKS:
    print(f'\n📍 Baseline: {task_name}')
    evaluator = Evaluator(task_name=task_name, policy=policy,
                          n_episodes=VLA_EPISODES, max_steps=MAX_STEPS)
    result = evaluator.evaluate()
    sr = result.get('success_rate', 0.0)
    vla_baseline[task_name] = {'success_rate': sr, **result}
    print(f'  SR: {sr:.1%}')

with open(BASELINE_DIR / 'vlabench_baseline.json', 'w') as f:
    json.dump(vla_baseline, f, indent=2)
print('\n✅ VLABench baseline done')

In [ ]:
# ── VLABench: Switch from baseline server to ablated server ───────────────
# Stop the baseline server, start serve_policy_ablated.py (state_proj zeroed
# server-side). Both servers use the same port — baseline must be stopped first.
import os, subprocess, socket, time

OPENPI_VLA    = '/content/openpi_vlabench'
VLABENCH_CKPT = '/content/ckpt_pi0_vlabench'

# Stop baseline server
server_proc.terminate()
server_proc.wait()
print('🛑 Baseline openpi VLABench server stopped')

# Start ablated server
server_proc_ablated = subprocess.Popen(
    ['uv', 'run', '--project', OPENPI_VLA,
     'python', f'{OPENPI_VLA}/scripts/serve_policy_ablated.py',
     '--env', 'vlabench',
     '--policy.config', 'pi0_ft_vlabench_primitive',
     '--policy.dir', VLABENCH_CKPT,
     '--port', str(SERVER_PORT)],
    cwd=OPENPI_VLA,
    env={**os.environ, 'XLA_PYTHON_CLIENT_MEM_FRACTION': '0.85'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f'🚀 Ablated openpi VLABench server started (PID {server_proc_ablated.pid}), '
      f'waiting for port {SERVER_PORT}...')

# Poll until server is ready
deadline = time.time() + 300
while time.time() < deadline:
    try:
        with socket.create_connection(('localhost', SERVER_PORT), timeout=1):
            break
    except OSError:
        time.sleep(3)
else:
    server_proc_ablated.kill()
    out, _ = server_proc_ablated.communicate()
    print('Server output:\n', out.decode()[-3000:])
    raise RuntimeError(f'Ablated openpi VLABench server did not open port {SERVER_PORT} within 5 minutes')

print(f'✅ Ablated openpi VLABench server ready on port {SERVER_PORT} (state_proj zeroed)')

In [ ]:
# ── VLABench Ablation ─────────────────────────────────────────────────────
# ABLATION METHOD: server-side weight zeroing of state_proj.
#
# serve_policy_ablated.py patches Policy.__init__ to zero state_proj.kernel
# and state_proj.bias before the JIT closure is compiled. flax.nnx reads
# .kernel.value at each call, so state_proj(x) = x @ 0 + 0 = 0 for all x.
#
# This is methodologically equivalent to the output-zeroing register_forward_hook
# used for LIBERO/MetaWorld — the state embedding reaching the expert model is
# identically zero regardless of the input proprioceptive state.
import json, sys

sys.path.insert(0, '/content/VLABench')
from VLABench.evaluation.model.policy.openpi import OpenPiPolicy
from VLABench.evaluation.evaluator import Evaluator

# Standard OpenPiPolicy — ablation is fully server-side
policy = OpenPiPolicy(host='localhost', port=SERVER_PORT)

vla_ablation = {}
for task_name in VLA_TASKS:
    print(f'\n📍 Ablation: {task_name}')
    evaluator = Evaluator(task_name=task_name, policy=policy,
                          n_episodes=VLA_EPISODES, max_steps=MAX_STEPS)
    result = evaluator.evaluate()
    sr = result.get('success_rate', 0.0)
    vla_ablation[task_name] = {'success_rate': sr, **result}
    print(f'  SR (ablated): {sr:.1%}')

with open(ABLATION_DIR / 'vlabench_ablation.json', 'w') as f:
    json.dump(vla_ablation, f, indent=2)
print('\n✅ VLABench ablation done')


In [ ]:
# ── VLABench: Shut down ablated openpi JAX server ────────────────────────
server_proc_ablated.terminate()
server_proc_ablated.wait()
print('🛑 Ablated openpi server stopped')
# JAX releases GPU memory when the process exits; flush any remaining alloc.
import torch
torch.cuda.empty_cache()
print('🔧 VRAM released after VLABench section')


In [ ]:
# ── Results Summary ───────────────────────────────────────────────────────
import json
from pathlib import Path

def load_json(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else {}

libero_base = load_json(BASELINE_DIR / 'libero_baseline.json')
mw_base     = load_json(BASELINE_DIR / 'metaworld_baseline.json')
vla_base    = load_json(BASELINE_DIR / 'vlabench_baseline.json')
libero_abl  = load_json(ABLATION_DIR / 'libero_ablation.json')
mw_abl      = load_json(ABLATION_DIR / 'metaworld_ablation.json')
vla_abl     = load_json(ABLATION_DIR / 'vlabench_ablation.json')

def mean_sr(d):
    rates = [v['success_rate'] for v in d.values() if 'success_rate' in v]
    return sum(rates)/len(rates) if rates else float('nan')

print('='*55)
print(f'{"Benchmark":<20} {"Baseline SR":>12} {"Ablated SR":>12} {"Drop":>8}')
print('='*55)
for label, base, abl in [('LIBERO',     libero_base, libero_abl),
                          ('MetaWorld',  mw_base,     mw_abl),
                          ('VLABench',   vla_base,    vla_abl)]:
    b, a = mean_sr(base), mean_sr(abl)
    drop = b - a
    print(f'{label:<20} {b:>11.1%} {a:>11.1%} {drop:>7.1%}')
print('='*55)
print('Low drop → state encoder underutilized (paper hypothesis ✅)')


---
# π₀ (3.3B) — Proprioceptive State Utilization Analysis

**Part 2: Standalone Gradient Analysis**  
Uses `lerobot/pi0` (generalist) to measure gradient flow through `state_proj`,
confirming architectural underutilization independent of benchmark-specific fine-tuning.

# π0 (3.3B) - Proprioceptive State Utilization Analysis

**Model**: Physical-Intelligence π0.5-DROID (3.3B parameters)  
**State Encoder**: `state_proj` - **Single Linear layer** (NOT multi-layer MLP)

## ⚠️ Architecture Correction
**Previous (Incorrect) Assumption**: Multipler encoder with separate layers  
**Actual Architecture** (from [openpi repo](https://github.com/Physical-Intelligence/openpi)):
- Vision: PaliGemma ViT encoder
- Language: PaliGemma Gemma model  
- **State**: Single Linear projection (`state_proj`)
- Expert: Separate Expert Gemma for flow matching
- Action: Diffusion-based via flow matching

## Hypothesis
**Proprioceptive state information is underutilized** - despite having a dedicated projection layer, the model may not meaningfully use robot state for action prediction.

## Experimental Design  
1. **Baseline**: Run normal state through state_proj → capture gradients
2. **Ablation**: Zero out robot_state → capture gradients  
3. **Compare**: Calculate gradient change percentage
4. **Verdict**:
   - <10% change = ❌ UNDERUTILIZED (state doesn't matter)
   - 10-30% change = ⚠️ PARTIALLY UTILIZED
   - >30% change = ✅ WELL UTILIZED (model relies on state)

## Key Metrics
- **Gradient norm on state_proj layer**
- **Gradient change % when state is ablated**

---

## 1. Environment Setup

In [ ]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab")
    !nvidia-smi
else:
    print("💻 Running locally")

In [ ]:
# On Colab, torch/numpy/matplotlib/pillow/sklearn are pre-installed.
# lerobot will handle its own dependencies cleanly.
# Only pre-install packages that might be missing:
!pip install -q einops imageio

# Install lerobot if not already installed (required if running Part 2 standalone,
# i.e. without having run Part 1's install cell first).
try:
    import lerobot  # noqa: F401
    print('✅ lerobot already installed')
except ImportError:
    print('Installing lerobot ...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'lerobot@git+https://github.com/huggingface/lerobot.git'],
                   check=True)
    print('✅ lerobot installed')

print('✅ Extra prerequisites ready')


### π0 (openpi) Specific Setup

**Official Repository**: `Physical-Intelligence/openpi`

**Architecture** (π0.5-DROID, 3.3B params):
- Vision: PaliGemma ViT encoder
- State: `state_proj` - Single Linear layer (NOT multi-layer encoder)
- Language: PaliGemma language model  
- Expert: Separate Gemma model for flow matching
- Action: Flow matching + diffusion-based

**Critical Requirements**:
```bash
# Install uv (modern Python package manager)
curl -LsSf https://astral.sh/uv/install.sh | sh

# OR using pip
pip install uv

# Install transformers with required patches
pip install transformers==4.53.2

# Clone openpi repository
git clone https://github.com/Physical-Intelligence/openpi.git
cd openpi
uv pip install -e .
```

**Model Loading**:
```python
from openpi.inference.policy import policy_config

# Load π0.5-DROID model
policy = policy_config.create_trained_policy(
    ckpt_path="droid-dataset",
    ckpt_step="latest"  
)
model = policy.model
```

**Note**: π0 uses **single Linear `state_proj` layer**, not a multi-layer encoder. See updated hooks for correct architecture.

## 3. Load π0 Model

**Official Loading Method** (Recommended):
```python
# Use openpi's official policy loading
sys.path.insert(0, 'openpi')
from openpi.inference.policy import policy_config

# Load π0.5-DROID (3.3B) with PyTorch support
policy = policy_config.create_trained_policy(
    ckpt_path="droid-dataset",  # or your checkpoint path
    ckpt_step="latest"
)
model = policy.model
```

**Corrected Architecture**:
- `paligemma`: PaliGemma model (vision + language encoder)
  - `model.vision_encoder`: ViT for image processing
  - `model.language_model`: Gemma for text processing
- `gemma_expert`: Separate Expert Gemma for flow matching
- `state_proj`: **Single Linear layer** (NOT multi-layer MLP!)
  - Maps robot state to embedding space
- `action_in_proj`, `action_out_proj`: Linear layers for action processing

**Previous Assumption (INCORRECT)**:
- ❌ Multi-layer `proprio_encoder` - **This doesn't exist!**
- ❌ Causal attention layers - Handled by standard transformer

**Actual Implementation**:
- ✅ Single Linear `state_proj` layer for state encoding
- ✅ PaliGemma handles vision + language fusion
- ✅ Expert Gemma performs flow matching for actions

**Note**: See updated `pi0_hooks.py` for corrected hook attachment targeting `state_proj`.

In [ ]:
import os, sys, torch
from lerobot.policies.pi0.modeling_pi0 import PI0Policy

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Load lerobot/pi0 from HuggingFace ──────────────────────────────────
# lerobot/pi0 = original π0 architecture (4 B params):
#   • PaliGemma vision-language backbone
#   • state_proj: nn.Linear(max_state_dim, expert_width)   ← what we study
#   • Expert Gemma for flow-matching action prediction
# This is NOT π0.5 (which has no state encoder at all).
print("\nDownloading lerobot/pi0 from HuggingFace (first run ~15 GB, cached after)...")
policy = PI0Policy.from_pretrained("lerobot/pi0")
policy = policy.to(device)
print("✅ lerobot/pi0 loaded")

# ── Locate the underlying PI0Pytorch model with state_proj ──────────────
model = None
for attr in ["model_pi0", "pi0", "model", "policy_model", "pi0_model"]:
    candidate = getattr(policy, attr, None)
    if candidate is not None and hasattr(candidate, "state_proj"):
        model = candidate
        print(f"✅ PI0Pytorch found at policy.{attr}")
        break

if model is None:
    # Fallback: search all Module children
    for name, mod in policy.named_modules():
        if hasattr(mod, "state_proj") and isinstance(mod.state_proj, torch.nn.Linear):
            model = mod
            print(f"✅ PI0Pytorch found via named_modules: {name}")
            break

if model is None:
    raise RuntimeError(
        "Could not find PI0Pytorch with state_proj. "
        "Check lerobot version or inspect policy.__dict__"
    )

# ── Confirm state_proj ──────────────────────────────────────────────────
sp = model.state_proj
print(f"\n✅ state_proj confirmed: nn.Linear({sp.in_features}, {sp.out_features})")
print(f"   Input: {sp.in_features}-dim padded robot state")
print(f"   Output: {sp.out_features}-dim expert embedding")

cfg = policy.config
print(f"\nPolicy config:")
print(f"  max_state_dim : {cfg.max_state_dim}")
print(f"  action_dim    : {cfg.action_dim}")
print(f"  chunk_size    : {cfg.chunk_size}")

try:
    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"\nModel parameters: {n_params:.2f}B")
except:
    pass


## 6. Prepare Sample Data

In [ ]:
# lerobot uses a standardised dict-based batch format.
# Keys follow the pattern "observation.<sensor>" and "action".
# Image keys are derived dynamically from the loaded checkpoint's config so
# this cell works correctly for both lerobot/pi0_old and lerobot/pi0_libero_base.
# ─────────────────────────────────────────────────────────────────────

cfg = policy.config
batch_size    = 1
max_state_dim = cfg.max_state_dim   # e.g. 32
action_dim    = cfg.action_dim      # e.g. 32
chunk_size    = cfg.chunk_size      # e.g. 50

# Dynamically discover image keys expected by this checkpoint
if hasattr(cfg, 'input_features'):
    # cfg.input_features values are PolicyFeature dataclass instances, not plain dicts.
    # Check the .type attribute directly; fall back to the key-name heuristic.
    img_keys = [
        k for k, v in cfg.input_features.items()
        if 'image' in k.lower() or
           (hasattr(v, 'type') and str(getattr(v, 'type', '')).upper() == 'VISUAL') or
           (isinstance(v, dict) and v.get('type') == 'VISUAL')
    ]
else:
    # Fallback: use the known keys for lerobot/pi0_old
    img_keys = ['observation.images.camera0',
                'observation.images.camera1',
                'observation.images.camera2']

print(f'Image keys detected: {img_keys}')

# Synthetic batch — random values are fine for gradient analysis.
# The gradient pattern reflects the model's *architectural* reliance
# on state_proj, independent of task-specific data distribution.
batch = {
    # Proprioceptive state (padded to max_state_dim)
    "observation.state": torch.randn(batch_size, max_state_dim, device=device),
    # Ground-truth actions for flow-matching loss
    "action": torch.randn(batch_size, chunk_size, action_dim, device=device),
}
# Add a zero-image tensor for each expected image key (224×224 RGB)
for k in img_keys:
    batch[k] = torch.zeros(batch_size, 3, 224, 224, device=device)

print("✅ Synthetic batch prepared for π0 gradient analysis")
for k, v in batch.items():
    print(f"  {k}: {v.shape}")
print("\nNote: Synthetic data → gradient magnitudes reflect architectural utilisation,")
print("       not task-specific learned correlations.")


## 7. Run Forward and Backward Pass

In [ ]:
# lerobot policy.forward(batch) → (loss, output_dict)
# loss is the flow-matching MSE loss used during Pi0 training.
policy.train()
policy.zero_grad()

print("Running π0 forward pass (flow-matching loss)...")
try:
    loss, output_dict = policy.forward(batch)
    loss = loss.mean() if loss.dim() > 0 else loss

    print("Running backward pass...")
    loss.backward()

    print(f"✅ Forward + backward completed")
    print(f"   Flow-matching loss: {loss.item():.4f}")
    if output_dict:
        print(f"   Output keys: {list(output_dict.keys())}")

except (TypeError, ValueError):
    # Older lerobot versions return only loss (no output_dict),
    # causing ValueError when unpacking into (loss, output_dict).
    loss = policy.forward(batch)
    if isinstance(loss, (tuple, list)):
        loss = loss[0]
    loss = loss.mean() if loss.dim() > 0 else loss
    loss.backward()
    print(f"✅ Forward + backward (single-return variant). Loss: {loss.item():.4f}")

except Exception as e:
    print(f"⚠️  policy.forward failed: {e}")
    print("   Trying direct PI0Pytorch model.forward...")
    from types import SimpleNamespace
    # Use the first image key discovered in the batch
    first_img_key = next((k for k in batch if 'image' in k.lower()), None)
    first_img = batch[first_img_key].unsqueeze(1) if first_img_key else \
                torch.zeros(batch_size, 1, 3, 224, 224, device=device)
    obs = SimpleNamespace(
        images=first_img,   # (B, n_cams, C, H, W)
        state=batch["observation.state"],
        tokenized_prompt=torch.zeros(batch_size, 20, dtype=torch.long, device=device),
        tokenized_prompt_mask=torch.ones(batch_size, 20, dtype=torch.bool, device=device),
    )
    actions = batch["action"]
    loss = model(obs, actions).mean()
    loss.backward()
    print(f"✅ Direct model.forward succeeded. Loss: {loss.item():.4f}")


## 8. Baseline: Capture Gradients with Normal State

In [ ]:
# ── Baseline: capture state_proj weight gradient after normal forward+backward ──
# The weight gradient tells us how strongly the loss signal propagates through
# state_proj, i.e. how much the network was relying on the state input.

grad = model.state_proj.weight.grad
if grad is not None:
    baseline_norm = grad.norm().item()
    print(f"✅ state_proj weight gradient norm (normal state): {baseline_norm:.6f}")
    print(f"   Weight shape: {model.state_proj.weight.shape}")
    print(f"   This is our baseline — model sees real robot state")
else:
    baseline_norm = 0.0
    print("⚠️ No gradient on state_proj.weight — did the forward+backward pass run?")
    print("   Re-run cell 7 before this cell.")

## 9. Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Zero the OUTPUT of `proprio_encoder`, not the input.

This directly tests: "What if the multi-layer encoder contributed nothing?"

In [ ]:
# ── Ablation: zero state_proj output, re-run forward+backward ─────────────────
# Attaching a forward hook that returns zeros for state_proj output means the
# downstream expert model receives no proprioceptive state information at all.
# We then check how much the weight gradient changes vs the baseline.

def zero_output_hook(module, input, output):
    """Replace state_proj output with zeros — model acts as if state is absent."""
    return torch.zeros_like(output)

ablation_handle = model.state_proj.register_forward_hook(zero_output_hook)
print(f"✓ Ablation hook on state_proj "
      f"({model.state_proj.in_features} → {model.state_proj.out_features})")
print(f"  Hook returns zeros → model acts as if state is absent")

# Re-run forward + backward with hook active
policy.zero_grad()
print("\nRunning ablation forward+backward (state_proj output = zeros)...")
try:
    loss_ablated, _ = policy.forward(batch)
    loss_ablated = loss_ablated.mean() if loss_ablated.dim() > 0 else loss_ablated
except (TypeError, ValueError):
    # Older lerobot versions return only loss (no output_dict),
    # causing ValueError when unpacking into (loss_ablated, _).
    loss_ablated = policy.forward(batch)
    if isinstance(loss_ablated, (tuple, list)):
        loss_ablated = loss_ablated[0]
    loss_ablated = loss_ablated.mean() if loss_ablated.dim() > 0 else loss_ablated
loss_ablated.backward()

# Remove hook immediately after backward
ablation_handle.remove()
print("✓ Ablation complete — hook removed")


## 10. Compare Gradients: Normal vs Ablation

In [ ]:
# ── Compare: ablated gradient norm vs baseline ─────────────────────────────────
grad_ablated = model.state_proj.weight.grad
if grad_ablated is not None:
    ablated_norm = grad_ablated.norm().item()
else:
    ablated_norm = 0.0

grad_change_pct = (abs(baseline_norm - ablated_norm) / baseline_norm * 100
                   if baseline_norm > 0 else 0.0)

print(f"{'='*60}")
print(f"GRADIENT COMPARISON")
print(f"{'='*60}")
print(f"Normal state   → state_proj ‖∇w‖: {baseline_norm:.6f}")
print(f"Ablated state  → state_proj ‖∇w‖: {ablated_norm:.6f}")
print(f"Change:                            {grad_change_pct:.1f}%")
print(f"{'='*60}")
print(f"\nLoss with normal state:   {loss.item():.4f}")
print(f"Loss with ablated state:  {loss_ablated.item():.4f}")
print(f"Loss Δ: {abs(loss.item() - loss_ablated.item()):.4f}")

## 11. Verdict: Is Proprioceptive State Utilized?

In [ ]:
# ── Verdict ────────────────────────────────────────────────────────────────────
if grad_change_pct < 10:
    verdict = "UNDERUTILIZED"
    explanation = ("When state_proj output is zeroed, weight gradients barely change (<10%). "
                   "The model does not rely meaningfully on the proprioceptive state encoder.")
elif grad_change_pct < 30:
    verdict = "PARTIALLY UTILIZED"
    explanation = ("Some gradient sensitivity to state removal (10–30%). "
                   "The dependency exists but is weak.")
else:
    verdict = "WELL UTILIZED"
    explanation = ("Strong gradient response (>30%) when state_proj output is ablated. "
                   "The model genuinely uses the proprioceptive state input.")

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"{'='*60}")
print(f"\nGradient change: {grad_change_pct:.1f}%")
print(f"\n{explanation}")
print(f"\nMethodology:")
print(f"  • Model: lerobot/pi0 (generalist, {sum(p.numel() for p in model.parameters())/1e9:.1f}B params)")
print(f"  • Layer: state_proj  nn.Linear({model.state_proj.in_features}, {model.state_proj.out_features})")
print(f"  • Loss:  flow-matching MSE (same objective as Pi0 training)")
print(f"  • Metric: ‖∇w‖ of state_proj.weight — normal vs output-ablated")
print(f"{'='*60}")

# ── Save to Google Drive (if available) ────────────────────────────────────────
import json, os
results = {
    'model': 'lerobot/pi0 (generalist)',
    'state_encoder': 'state_proj  nn.Linear',
    'state_proj_in': model.state_proj.in_features,
    'state_proj_out': model.state_proj.out_features,
    'ablation_method': 'zero_state_proj_output',
    'baseline_grad_norm': float(baseline_norm),
    'ablated_grad_norm': float(ablated_norm),
    'gradient_change_pct': float(grad_change_pct),
    'baseline_loss': float(loss.item()),
    'ablated_loss': float(loss_ablated.item()),
    'verdict': verdict,
}

# Save locally
local_path = 'pi0_gradient_results.json'
with open(local_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✓ Results saved locally: {local_path}")

# Save to Drive if mounted
drive_gradient_dir = '/content/drive/MyDrive/pi0_study/Results/gradient'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_gradient_dir, exist_ok=True)
    drive_path = f'{drive_gradient_dir}/pi0_gradient_results.json'
    with open(drive_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Results saved to Drive: {drive_path}")